# Frequent Itemsets: Sampling and SON Algorithm

## Learning Objectives

1. **Explain** why random sampling can speed up frequent itemset mining
2. **Implement** the Savasere-Omiecinski-Navathe (SON) algorithm
3. **Analyze** the false positive / false negative trade-off for the threshold
4. **Describe** how SON maps to the MapReduce model

## The Sampling Approach

**Core idea:** instead of reading the full dataset twice, read a random sample $S$ once.

1. Draw a sample $S$ of size $|S| = p \cdot |B|$ from baskets $B$
2. Find itemsets frequent in $S$ at threshold $p \cdot s$ (scaled proportionally)
3. Make a single pass through the full data to verify candidates

**Problem:**
- An itemset frequent in $B$ may not be frequent in $S$ → **false negatives**
- Use threshold $p \cdot s \cdot (1 - \epsilon)$ to reduce false negatives (but increases false positives)

**Trade-off:** lower threshold → fewer false negatives, more false positives to verify.

## The SON Algorithm (Savasere-Omiecinski-Navathe)

SON eliminates false negatives *entirely* and is MapReduce-friendly.

**Key theorem:** An itemset $X$ can be frequent in $B$ only if it is frequent in **at least one** chunk of $B$.

**Algorithm:**
1. Divide baskets into $k$ chunks of size $\approx |B|/k$ (fit in memory)
2. **Pass 1:** For each chunk $C_i$, find itemsets frequent in $C_i$ at threshold $s \cdot |C_i|/|B|$. Union all candidate sets.
3. **Pass 2:** Count candidates over the full dataset; keep those with support ≥ $s$.

**Correctness:** If $X$ is frequent in $B$, it must be frequent in some chunk → always in the candidate set → checked in pass 2. Zero false negatives.

**Cost:** Pass 1 scales with chunk size (fits in memory); pass 2 counts only verified candidates.

In [ ]:
from collections import defaultdict
from itertools import combinations
import random

def apriori_in_memory(baskets, min_count, max_k=3):
    """Find frequent itemsets in a list of baskets using Apriori."""
    n = len(baskets)
    freq = {}
    # 1-itemsets
    count1 = defaultdict(int)
    for b in baskets:
        for i in b: count1[frozenset([i])] += 1
    freq[1] = {k:v for k,v in count1.items() if v >= min_count}
    # k-itemsets
    for k in range(2, max_k+1):
        prev_freq = list(freq[k-1].keys())
        count_k = defaultdict(int)
        for b in baskets:
            b_set = frozenset(b)
            for candidate in combinations(sorted(set().union(*prev_freq)), k):
                c = frozenset(candidate)
                if c.issubset(b_set):
                    # Prune: all (k-1)-subsets must be frequent
                    if all(frozenset(s) in freq[k-1] for s in combinations(candidate, k-1)):
                        count_k[c] += 1
        freq[k] = {k2:v for k2,v in count_k.items() if v >= min_count}
        if not freq[k]: break
    return freq

def son_algorithm(baskets, s, n_chunks=3):
    """SON: two-pass algorithm safe against false negatives."""
    n = len(baskets)
    global_threshold = s * n

    # Partition baskets into chunks
    chunk_size = n // n_chunks
    chunks = [baskets[i*chunk_size:(i+1)*chunk_size] for i in range(n_chunks)]
    # Add remainder to last chunk
    if n % n_chunks: chunks[-1] += baskets[n_chunks*chunk_size:]

    # Pass 1: find local frequent itemsets in each chunk
    candidates = set()
    for chunk in chunks:
        local_threshold = s * len(chunk)   # proportional threshold
        local_freq = apriori_in_memory(chunk, local_threshold)
        for k_freq in local_freq.values():
            candidates |= k_freq.keys()
    print(f"Candidate itemsets from pass 1: {len(candidates)}")

    # Pass 2: count candidates over entire dataset
    global_count = defaultdict(int)
    for b in baskets:
        b_set = frozenset(b)
        for c in candidates:
            if c.issubset(b_set):
                global_count[c] += 1
    freq_global = {c:cnt for c,cnt in global_count.items() if cnt >= global_threshold}
    return freq_global

# Generate test data
rng = random.Random(42)
items = list('ABCDEFGH')
baskets = [[rng.choice(items) for _ in range(rng.randint(2,5))] for _ in range(300)]

s = 0.05
result = son_algorithm(baskets, s, n_chunks=3)
print(f"
Frequent itemsets (s={s}): {len(result)}")
for itemset, cnt in sorted(result.items(), key=lambda x:-x[1])[:10]:
    print(f"  {set(itemset)}: support={cnt/len(baskets):.3f}")

In [ ]:
# Verify: compare SON with exact Apriori on full dataset
exact = apriori_in_memory(baskets, min_count=s*len(baskets))
exact_all = set()
for k_freq in exact.values():
    exact_all |= k_freq.keys()

son_keys = set(result.keys())
print(f"Exact frequent itemsets: {len(exact_all)}")
print(f"SON frequent itemsets:   {len(son_keys)}")
print(f"False negatives (in exact, not in SON): {len(exact_all - son_keys)}")
print(f"False positives (in SON, not in exact): {len(son_keys - exact_all)}")
print(f"
SON is correct: {exact_all == son_keys}")

## SON in MapReduce

**Map phase (Pass 1):**
- Each mapper receives a chunk of baskets
- Outputs: `(itemset, 0)` for each locally frequent itemset (0 = candidate marker)

**Reduce phase (Pass 1):**
- Collect all candidates → broadcast to all mappers

**Map phase (Pass 2):**
- Each mapper counts candidates in its chunk
- Outputs: `(itemset, count)`

**Reduce phase (Pass 2):**
- Sum counts across chunks; emit if total count ≥ $s \cdot |B|$

This is naturally parallel: each chunk fits in mapper memory, and SON uses exactly 2 MapReduce jobs.